In [9]:
import sys
import os
import torch
import chromadb
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import json

project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.append(project_root)
    
from config import Config

# --- GPU Check ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Initializing embeddings on: {device.upper()}")

# Initialize embeddings with the dynamic device
embeddings = HuggingFaceEmbeddings(
    model_name=Config.EMBEDDING_MODEL,
    model_kwargs={'device': device}
)

# Initialize Vector Store
vs = Chroma(
    persist_directory=str(Config.DATA_DIR / "chroma_db"),
    embedding_function=embeddings
)


[*] Initializing embeddings on: CUDA


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1282.56it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
queries = [
    "I am currently invisible and want to shoot my bow at the guard. Do I get advantage?",
    "My fighter wants to grab the cultist and wrestle him to the ground. How does that work?",
    "I want to cast Magic Missile. Does it require an attack roll?",
    "What happens to my character if I drop to exactly 0 hit points?",
    "I try to pickpocket Arin's coin purse without him noticing.",             
    "I attempt to shove the bandit captain backwards into the fireplace.",      
    "I'm concentrating on Bless and I want to cast Invisibility on myself. Can I do both?",
    "I want to hide behind the bar mid-combat. Do I need to take an action for that?",
    "I try to run away from the enemy but I don't want to get hit on the way out. What do I do?",
    "The enemy walks away from me mid-combat. Can I attack them as they leave?",
    "I fell off the chandelier, about 30 feet up. How much damage do I take?",
    "I want to ready my crossbow and shoot the first person who walks through the door.",
    "I want to help my ally hit the bandit. Can I do something to give them a better chance?",
    "I'm going to dodge this turn instead of attacking. What exactly does that give me?",
    "The enemy knocked me prone. How do I stand back up and can I still attack this turn?",
    "I want to attack with my sword and also cast Fireball in the same turn. Can I?",
    "I want to wield a shortsword in each hand and attack with both. How does that work?",
    "My barbarian wants to swing wildly to guarantee a hit. Is there a way to do that?",
    "I successfully grappled the bandit. Can I drag him across the tavern now?",
    "My character just rolled their third failed death saving throw. What happens?",
]


In [11]:
benchmark = []

for i, query in enumerate(queries):
    results = vs.similarity_search(query, k=10)
    
    benchmark.append({
        "query_id": i + 1,
        "query": query,
        "retrieved_chunks": [
            {
                "rank": j + 1,
                "source": doc.metadata.get('header_chain', 'No Header'),
                "content": doc.page_content
            }
            for j, doc in enumerate(results)
        ]
    })
    print(f"[{i+1}/20] Done: {query[:60]}...")

output_path = os.path.join(project_root, "data", "retrieval_browse.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(benchmark, f, indent=4)

print(f"\nSaved to {output_path}")

[1/20] Done: I am currently invisible and want to shoot my bow at the gua...
[2/20] Done: My fighter wants to grab the cultist and wrestle him to the ...
[3/20] Done: I want to cast Magic Missile. Does it require an attack roll...
[4/20] Done: What happens to my character if I drop to exactly 0 hit poin...
[5/20] Done: I try to pickpocket Arin's coin purse without him noticing....
[6/20] Done: I attempt to shove the bandit captain backwards into the fir...
[7/20] Done: I'm concentrating on Bless and I want to cast Invisibility o...
[8/20] Done: I want to hide behind the bar mid-combat. Do I need to take ...
[9/20] Done: I try to run away from the enemy but I don't want to get hit...
[10/20] Done: The enemy walks away from me mid-combat. Can I attack them a...
[11/20] Done: I fell off the chandelier, about 30 feet up. How much damage...
[12/20] Done: I want to ready my crossbow and shoot the first person who w...
[13/20] Done: I want to help my ally hit the bandit. Can I do something to